In [50]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score


In [51]:
%pip install pandas numpy matplotlib seaborn scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [52]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [53]:
import pandas as pd
df = pd.read_excel(r"C:\Users\ronin\Documents\Personal Projects\Flight-Price-Prediction\data\Data_Train.xlsx")

In [54]:
df.head
df.info()
df.isna().sum()
df = df.dropna()

<class 'pandas.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Airline          10683 non-null  str  
 1   Date_of_Journey  10683 non-null  str  
 2   Source           10683 non-null  str  
 3   Destination      10683 non-null  str  
 4   Route            10682 non-null  str  
 5   Dep_Time         10683 non-null  str  
 6   Arrival_Time     10683 non-null  str  
 7   Duration         10683 non-null  str  
 8   Total_Stops      10682 non-null  str  
 9   Additional_Info  10683 non-null  str  
 10  Price            10683 non-null  int64
dtypes: int64(1), str(10)
memory usage: 918.2 KB


In [55]:
df[['Journey_day', 'Journey_month']] = df['Date_of_Journey'].str.split('/', expand=True).iloc[:,0:2].astype(int)
df = df.drop(columns='Date_of_Journey')

df['Duration'] = pd.to_timedelta(df['Duration'])
df['Duration'] = df['Duration'] / np.timedelta64(1, 'm')
df['Duration'] = df['Duration'].astype(int)

df[['Dep_hour', 'Dep_min']] = df['Dep_Time'].str.split(':', expand=True).astype(int)
df = df.drop(columns='Dep_Time')
df[['Arrival_hour', 'Arrival_min']] = df['Arrival_Time'].str.extract(r'(\d{2}):(\d{2})')
df['Arrival_hour'] = df['Arrival_hour'].astype(int)
df['Arrival_min'] = df['Arrival_min'].astype(int)
df = df.drop(columns='Arrival_Time')

mapping_dict = {'1 stop': 1, '2 stops': 2, '3 stops': 3, '4 stops' : 4, "non-stop" : 0}
df['Total_Stops'] = df['Total_Stops'].map(mapping_dict)
df = df.drop(columns = ['Route', 'Additional_Info'])

In [56]:
df.head

<bound method NDFrame.head of            Airline    Source Destination  Duration  Total_Stops  Price  \
0           IndiGo  Banglore   New Delhi       170            0   3897   
1        Air India   Kolkata    Banglore       445            2   7662   
2      Jet Airways     Delhi      Cochin      1140            2  13882   
3           IndiGo   Kolkata    Banglore       325            1   6218   
4           IndiGo  Banglore   New Delhi       285            1  13302   
...            ...       ...         ...       ...          ...    ...   
10678     Air Asia   Kolkata    Banglore       150            0   4107   
10679    Air India   Kolkata    Banglore       155            0   4145   
10680  Jet Airways  Banglore       Delhi       180            0   7229   
10681      Vistara  Banglore   New Delhi       160            0  12648   
10682    Air India     Delhi      Cochin       500            2  11753   

       Journey_day  Journey_month  Dep_hour  Dep_min  Arrival_hour  \
0          

In [57]:
X = df.drop(columns='Price')
y = df['Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [58]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 8545 entries, 10005 to 7270
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Airline        8545 non-null   str  
 1   Source         8545 non-null   str  
 2   Destination    8545 non-null   str  
 3   Duration       8545 non-null   int64
 4   Total_Stops    8545 non-null   int64
 5   Journey_day    8545 non-null   int64
 6   Journey_month  8545 non-null   int64
 7   Dep_hour       8545 non-null   int64
 8   Dep_min        8545 non-null   int64
 9   Arrival_hour   8545 non-null   int64
 10  Arrival_min    8545 non-null   int64
dtypes: int64(8), str(3)
memory usage: 801.1 KB


In [59]:
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

categorical_cols = ['Destination', 'Source', 'Airline']
preprocessor = ColumnTransformer(
    transformers = [('cat', OneHotEncoder(handle_unknown="ignore"), categorical_cols)],
    remainder= "passthrough"
)

pipeline = Pipeline(steps= [
    ('preprocessor',preprocessor), 
    ('model', RandomForestRegressor())
    ])

pipeline.fit(X_train, y_train)

joblib.dump(pipeline, 'flight_pipeline.pkl')

['flight_pipeline.pkl']

In [60]:
y_pred = pipeline.predict(X_test)
meanAbsoluteError = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test,y_pred)
print(meanAbsoluteError, r2)

1157.9623939714681 0.8197556800040943


In [61]:
param_grid = [{'model__n_estimators': [100, 200], 'model__max_depth': [None, 10, 20]}]
grid = GridSearchCV(pipeline, param_grid)
grid.fit(X_train, y_train)
print(grid.best_params_)

{'model__max_depth': 10, 'model__n_estimators': 200}


In [62]:
y_pred = grid.predict(X_test)
meanAbsoluteError2 = mean_absolute_error(y_test, y_pred)
r2_2 = r2_score(y_test,y_pred)
print(meanAbsoluteError, r2)

1157.9623939714681 0.8197556800040943


In [63]:
import pandas as pd
new_flight = pd.DataFrame([X_test.iloc[0]])
new_flight['Total_Stops'] = 3
grid.predict(new_flight)

array([16666.09386699])

In [64]:
X_train.columns

Index(['Airline', 'Source', 'Destination', 'Duration', 'Total_Stops',
       'Journey_day', 'Journey_month', 'Dep_hour', 'Dep_min', 'Arrival_hour',
       'Arrival_min'],
      dtype='str')